# Collecting data

## Earthquake event catalog


Best method to collect data is to use their API to get chunks of data in `GeoJSON` format before converting them to one large dataframe and store it as a parquet file 

ANSS Comprehensive Earthquake Catalog (ComCat) Documentation: [https://earthquake.usgs.gov/data/comcat](https://earthquake.usgs.gov/data/comcat)
  
API documentation: [https://earthquake.usgs.gov/fdsnws/event/1/](https://earthquake.usgs.gov/fdsnws/event/1/)

In [1]:
from pathlib import Path
import json
import calendar
from datetime import datetime, timedelta
import requests


BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"


def fetch_comcat_chunk(
    starttime: str,
    endtime: str,
    minmagnitude: float = 2.5,
    offset: int = 1,
    limit: int = 20000,
    eventtype: str = "earthquake",
    orderby: str = "time-asc",
):
    """
    Fetch one ComCat chunk from the API and return the GeoJSON response as dict.
    starttime/endtime should be strings like '2018-01-01' or ISO datetime strings.
    """
    params = {
        "format": "geojson",
        "eventtype": eventtype,
        "starttime": starttime,
        "endtime": endtime,
        "minmagnitude": minmagnitude,
        "orderby": orderby,
        "limit": limit,
        "offset": offset,
    }

    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()


def save_raw_geojson(data: dict, filepath: Path):
    """
    Save a GeoJSON dict to disk.
    """
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f)


def month_ranges(start_date: str, end_date: str):
    """
    Yield monthly date ranges covering [start_date, end_date].

    Returns tuples:
        (chunk_start_str, chunk_end_str, year, month)

    Example:
        ('2018-01-01', '2018-01-31', 2018, 1)
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()

    if start > end:
        raise ValueError("start_date must be <= end_date")

    current = start.replace(day=1)

    while current <= end:
        year = current.year
        month = current.month
        last_day = calendar.monthrange(year, month)[1]

        chunk_start = max(start, current)
        chunk_end = min(end, current.replace(day=last_day))

        yield (
            chunk_start.isoformat(),
            chunk_end.isoformat(),
            year,
            month,
        )

        if month == 12:
            current = current.replace(year=year + 1, month=1, day=1)
        else:
            current = current.replace(month=month + 1, day=1)


def download_comcat_range(
    start_date: str,
    end_date: str,
    output_dir: str = "data/raw/comcat",
    minmagnitude: float = 2.5,
    refresh: bool = False,
    verbose: bool = True,
):
    """
    Download ComCat data month by month over a user-specified date range.

    - If a month's GeoJSON file already exists, skip it unless refresh=True.
    - Saves one GeoJSON file per month.

    Returns:
        list[Path] of all expected filepaths
    """
    output_dir = Path(output_dir)
    saved_files = []

    for chunk_start, chunk_end, year, month in month_ranges(start_date, end_date):
        year_dir = output_dir / str(year)
        filename = f"comcat_{year}_{month:02d}.geojson"
        filepath = year_dir / filename

        saved_files.append(filepath)

        if filepath.exists() and not refresh:
            if verbose:
                print(f"[SKIP] Already exists: {filepath}")
            continue

        if verbose:
            print(f"[FETCH] {chunk_start} to {chunk_end} -> {filepath}")

        data = fetch_comcat_chunk(
            starttime=chunk_start,
            endtime=chunk_end,
            minmagnitude=minmagnitude,
        )
        save_raw_geojson(data, filepath)

        if verbose:
            n = len(data.get("features", []))
            print(f"[DONE] Saved {n} events to {filepath}")

    return saved_files

    

In [ ]:
# downloading all the comcat data in geojson format
file_paths = download_comcat_range(
    start_date="2015-01-01",
    end_date="2025-12-31",
    output_dir="data/raw/comcat",
    minmagnitude=2.5,
    refresh=False,
)

# ComCat GeoJSON — concise descriptive metadata

This file is a **GeoJSON FeatureCollection** of earthquake events returned by the USGS/ANSS ComCat query.

## Top-level structure

### `type`
- Usually `"FeatureCollection"`
- Indicates that the file contains a collection of geospatial features

### `metadata`
- Information about the query result itself, not about individual earthquakes
- Common uses:
  - dataset title
  - API status / response code
  - generated timestamp
  - result count
  - API URL used for the query

### `features`
- A list of earthquake event records
- **Each item in `features` = one earthquake event**

### `bbox`
- Bounding box for all returned events
- Usually in the form:
  - `[min_longitude, min_latitude, min_depth, max_longitude, max_latitude, max_depth]`
- Describes the overall spatial extent of the returned data

---

## Structure of each event in `features`

Each event contains 4 main parts:

### `type`
- Usually `"Feature"`
- Standard GeoJSON label for one feature/object

### `id`
- Unique ComCat event identifier
- Best used as the event’s primary key

### `geometry`
- Spatial location of the event

#### `geometry.type`
- Usually `"Point"`
- Means the earthquake is represented as a single location point

#### `geometry.coordinates`
- A 3-value list:
  - `[longitude, latitude, depth]`
- Meaning:
  - index `0`: longitude
  - index `1`: latitude
  - index `2`: depth in km

### `properties`
- Non-spatial metadata about the earthquake
- This is where most of the useful tabular fields are stored

---

## `properties` fields in your sample

### Event size / seismic measurement

#### `mag`
- Earthquake magnitude
- Numeric event strength value

#### `magType`
- Magnitude type
- Examples may include:
  - `ml`
  - `mb`
  - `mw`
  - `mwr`
- Important because magnitudes from different scales are not exactly the same

---

### Event timing

#### `time`
- Event origin time
- Stored as Unix timestamp in **milliseconds**
- This is the main event occurrence time

#### `updated`
- Last update time for the event record
- Also in Unix timestamp milliseconds
- Useful to know whether the record was revised later

#### `tz`
- Time zone offset field
- Often null or less important for analysis
- Usually you should rely on `time` and convert to UTC datetime

---

### Event location / description

#### `place`
- Human-readable description of location
- Example style:
  - `"74km NW of Kota Ternate, Indonesia"`

#### `title`
- Human-readable event title
- Usually combines magnitude and place
- Example style:
  - `"M 4.6 - 74km NW of Kota Ternate, Indonesia"`

---

### Event links / identifiers

#### `url`
- Web page for the event summary

#### `detail`
- API endpoint with more detailed event information
- Useful if you later want richer metadata or products

#### `code`
- Event code within the reporting network

#### `net`
- Reporting seismic network code
- Identifies the source network/provider

#### `ids`
- Comma-separated list of associated event IDs
- May include IDs from different contributing systems

#### `sources`
- Comma-separated list of source networks contributing to the event record

#### `types`
- Comma-separated list of product/data types available for the event
- Can indicate what additional information exists for that event

---

### Event quality / reliability / intensity-related fields

#### `status`
- Review status of the event
- Often useful to distinguish:
  - automatically generated records
  - reviewed records

#### `sig`
- Significance score
- A general event significance measure used by USGS

#### `nst`
- Number of seismic stations used
- Can be useful as a rough quality indicator

#### `dmin`
- Distance to nearest station
- Smaller values may indicate better local constraint

#### `rms`
- Root mean square of travel-time residuals
- Often used as a location-quality indicator

#### `gap`
- Azimuthal gap
- Describes station coverage around the event
- Lower values are often better

---

### Shaking / public impact / hazard fields

#### `felt`
- Number of felt reports submitted by people
- Can be null if unavailable

#### `cdi`
- Community Determined Intensity
- Public-reported shaking intensity estimate

#### `mmi`
- Modified Mercalli Intensity
- Instrumental or estimated shaking intensity

#### `alert`
- Alert level for hazard impact
- Can be null or values like color-coded alert categories

#### `tsunami`
- Tsunami flag
- Usually indicates whether the event was flagged for tsunami consideration

---

### Event classification

#### `type`
- Event type
- Often `"earthquake"`
- Useful in case other event types appear in a broader query

---

## What matters most for your project

For your aftershock prediction task, the most important fields are:

- `id`
- `time`
- `geometry.coordinates[0]` → longitude
- `geometry.coordinates[1]` → latitude
- `geometry.coordinates[2]` → depth
- `mag`
- `magType`
- `status`
- `gap`
- `dmin`
- `rms`
- `nst`
- `net`
- `type`

---

## Best way to think about this file

- `metadata` = information about the query result
- `features` = all earthquake events returned
- one item in `features` = one earthquake
- `geometry` = where it happened
- `properties` = what happened and supporting metadata

In [6]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd


RECOMMENDED_COLUMNS = [
    "event_id",
    "time",
    "updated",
    "latitude",
    "longitude",
    "depth_km",
    "magnitude",
    "magnitude_type",
    "status",
    "event_type",
    "net",
    "gap",
    "dmin",
    "rms",
    "nst",
]


def comcat_geojson_to_dataframe(filepath: str | Path) -> pd.DataFrame:
    """
    Read one ComCat GeoJSON file and convert it into a flat dataframe.

    Parameters
    ----------
    filepath : str | Path
        Path to one ComCat GeoJSON file.

    Returns
    -------
    pd.DataFrame
        Dataframe with one row per earthquake event.
    """
    filepath = Path(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    features = data.get("features", [])
    rows = []

    for feature in features:
        props = feature.get("properties", {})
        geom = feature.get("geometry", {})
        coords = geom.get("coordinates", [None, None, None])

        longitude = coords[0] if len(coords) > 0 else None
        latitude = coords[1] if len(coords) > 1 else None
        depth_km = coords[2] if len(coords) > 2 else None

        row = {
            "event_id": feature.get("id"),
            "time": props.get("time"),
            "updated": props.get("updated"),
            "latitude": latitude,
            "longitude": longitude,
            "depth_km": depth_km,
            "magnitude": props.get("mag"),
            "magnitude_type": props.get("magType"),
            "status": props.get("status"),
            "event_type": props.get("type"),
            "net": props.get("net"),
            "gap": props.get("gap"),
            "dmin": props.get("dmin"),
            "rms": props.get("rms"),
            "nst": props.get("nst"),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame(columns=RECOMMENDED_COLUMNS)

    # convert timestamps from unix milliseconds to datetime
    df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True, errors="coerce")
    df["updated"] = pd.to_datetime(df["updated"], unit="ms", utc=True, errors="coerce")

    # enforce column order
    df = df[RECOMMENDED_COLUMNS]

    # optional: sort within file
    df = df.sort_values("time").reset_index(drop=True)

    return df


def month_start(dt: datetime) -> datetime:
    return dt.replace(day=1)


def next_month(dt: datetime) -> datetime:
    if dt.month == 12:
        return dt.replace(year=dt.year + 1, month=1, day=1)
    return dt.replace(month=dt.month + 1, day=1)


def iter_months(start_date: str, end_date: str):
    """
    Yield (year, month) pairs from start_date to end_date inclusive.
    Dates must be YYYY-MM-DD.
    """
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")

    if start_dt > end_dt:
        raise ValueError("start_date must be <= end_date")

    current = month_start(start_dt)

    while current <= end_dt:
        yield current.year, current.month
        current = next_month(current)


def build_comcat_filepath(
    root_dir: str | Path,
    year: int,
    month: int,
) -> Path:
    """
    Build filepath like:
    data/raw/comcat/2015/comcat_2015_01.geojson
    """
    root_dir = Path(root_dir)
    return root_dir / str(year) / f"comcat_{year}_{month:02d}.geojson"


def load_comcat_date_range(
    start_date: str,
    end_date: str,
    root_dir: str | Path = "data/raw/comcat",
    skip_missing: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Load and combine all ComCat GeoJSON files whose month falls within
    the provided date range.

    Parameters
    ----------
    start_date : str
        YYYY-MM-DD
    end_date : str
        YYYY-MM-DD
    root_dir : str | Path
        Root directory containing yearly ComCat folders.
    skip_missing : bool
        If True, skip missing files. If False, raise FileNotFoundError.
    verbose : bool
        If True, print progress.

    Returns
    -------
    pd.DataFrame
        Combined dataframe filtered to the exact start/end date.
    """
    frames = []

    for year, month in iter_months(start_date, end_date):
        filepath = build_comcat_filepath(root_dir, year, month)

        if not filepath.exists():
            if skip_missing:
                if verbose:
                    print(f"[SKIP] Missing file: {filepath}")
                continue
            raise FileNotFoundError(f"Missing file: {filepath}")

        if verbose:
            print(f"[LOAD] {filepath}")

        df_month = comcat_geojson_to_dataframe(filepath)
        frames.append(df_month)

    if not frames:
        return pd.DataFrame(columns=RECOMMENDED_COLUMNS)

    df = pd.concat(frames, ignore_index=True)

    # filter to exact requested date range
    start_ts = pd.Timestamp(start_date, tz="UTC")
    end_ts = pd.Timestamp(end_date, tz="UTC") + pd.Timedelta(days=1)

    df = df[(df["time"] >= start_ts) & (df["time"] < end_ts)].copy()

    df = df.sort_values("time").reset_index(drop=True)

    return df

In [10]:
# Read one file to test the parsing
df_jan_2015 = comcat_geojson_to_dataframe("data/raw/comcat/2015/comcat_2015_01.geojson")
# print(df_jan_2015.head())
print(df_jan_2015.shape)

(2071, 15)


In [11]:
# load dataset across a date range
df_2015_2024 = load_comcat_date_range(
    start_date="2015-01-01",
    end_date="2024-12-31",
    root_dir="data/raw/comcat",
    skip_missing=True,
    verbose=False,
)

# print(df_2015_2024.head())
print(df_2015_2024.shape)


# Save the combined dataframe to Parquet for future use
# df_2015_2024.to_parquet("data/processed/comcat_2015.parquet", index=False)


(272048, 15)


# ComCat Dataset Overview

The ANSS Comprehensive Earthquake Catalog (ComCat) is a seismic event catalog that records earthquake occurrences together with their key physical, spatial, temporal, and quality-related attributes.  
For this project, ComCat serves as the **core event-level dataset** because it provides the essential information needed to identify:

- when an earthquake happened
- where it happened
- how large it was
- whether it can be treated as a candidate mainshock or aftershock
- how reliable the event record is

Each row in the cleaned tabular dataset represents **one earthquake event**.  
This allows us to sort events over time, measure distances between earthquakes, identify mainshock–aftershock pairs, and construct the prediction target:

**Given a mainshock of $M \ge 4.0$, predict how many hours until the next $M \ge 2.5$ aftershock occurs within 50 km, capped at 720 hours.**

## Selected Features and Why They Were Chosen

- `event_id`
  - Unique identifier for each earthquake event
  - Useful for deduplication, traceability, and linking records across processing steps

- `time`
  - Timestamp of when the earthquake occurred
  - Essential for ordering events and computing the target variable: time until the next qualifying aftershock

- `updated`
  - Timestamp of when the event record was last revised
  - Useful for understanding whether event information was later corrected or refined

- `latitude`
  - Geographic latitude of the earthquake epicenter
  - Needed to compute spatial distance between events

- `longitude`
  - Geographic longitude of the earthquake epicenter
  - Needed together with latitude to determine whether a later event is within 50 km of the mainshock

- `depth_km`
  - Depth of the earthquake in kilometers
  - May influence aftershock behaviour and provides additional physical context about the event

- `magnitude`
  - Numeric measure of earthquake size
  - Essential for identifying mainshocks ($M \ge 4.0$) and candidate aftershocks ($M \ge 2.5$)

- `magnitude_type`
  - Type of magnitude measurement used (for example, local magnitude or moment magnitude)
  - Helps account for differences in magnitude scales across events

- `status`
  - Indicates whether the event was automatically generated or reviewed
  - Useful as a proxy for record reliability and data quality

- `event_type`
  - Describes the type of seismic event
  - Helps ensure only true earthquake events are used in the analysis

- `net`
  - Reporting seismic network code
  - Useful for traceability and for capturing possible differences across contributing networks

- `gap`
  - Azimuthal gap in station coverage around the event
  - Acts as a quality indicator for how well the event location was constrained

- `dmin`
  - Distance to the nearest seismic station
  - Useful as a data quality signal, since events recorded farther from stations may be less precise

- `rms`
  - Root mean square of travel-time residuals
  - Another quality-related measure indicating how well the event solution fits the observations

- `nst`
  - Number of seismic stations used in the event solution
  - Can serve as a proxy for confidence in the event’s estimated parameters

## Summary

The selected ComCat features were chosen for three main reasons:

- **Target construction**  
  Needed to define mainshocks, aftershocks, time differences, and 50 km spatial constraints

- **Predictive value**  
  Provide physical and temporal information that may influence aftershock timing

- **Data quality control**  
  Help identify whether an event record is reliable enough to be used for modelling